In [2]:
import torch, transformers
import json

model_8B_id = "meta-llama/Meta-Llama-3-8B-Instruct"
model_3B_id = "meta-llama/Llama-3.2-3B-Instruct"
tok = transformers.AutoTokenizer.from_pretrained(model_3B_id)
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_3B_id, torch_dtype=torch.float16, device_map="cuda"
)

d:\GeoTKG\llama_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.80s/it]


In [45]:
def get_test_data():
    import json
    examples = []
    with open("D:\\GeoTKG\\cleandata\\tie\\test.json", "r") as f:
        examples=[json.loads(line) for line in f]
    return examples

def get_prompt(text, dct):
    sys = '''
    You are an information extraction system for geoscience and general texts. 
    Extract events and their temporal relations with arguments and time bounds.

    Rules:
    - Be literal, evidence-based; no invention. Use null if uncertain.
    - Resolve cross-sentence references (“this uplift”, “it”, etc.).
    - Return ONLY valid JSON, no markdown or commentary.

    Temporal relations (allowed values): BEFORE, AFTER, DURING, CONTAINS, IDENTITY, EQUALS, OVERLAPS.  
    - BEFORE: event1 ends before event2 starts  
    - AFTER: event1 starts after event2 ends  
    - DURING: event1 occurs within event2  
    - CONTAINS: event1 fully contains event2  
    - IDENTITY: same event
    - OVERLAPS: partial intersection
    - EQUALS: same time span, different events

    Time normalization:
    - Use explicit ISO if present (e.g., “1998-06”, “-0120/0010”).  
    - Normalize named geological periods/epochs (e.g., “Late Cretaceous”) to their standard numeric age ranges (e.g., start_time="100.5 Ma", end_time="66 Ma") using canonical ICS chronostratigraphic boundaries. 
    - If only order is known, leave times null.  
    - If interval given (e.g., “72-66 Ma” or “2017-2018 season”), set start_time/end_time as written.
    - Use the document creation time if anchor time is vague.

    Event arguments:
    - subject = agent/undergoer (or null)  
    - object = patient/theme (or null)  
    - event = short verb phrase (“uplift occurred”)  
    - start_time/end_time = normalized as above  
    - Keep spans concise and faithful

    Event identification:
    - Treat each distinct geologic or contemporary process/change as an event.  
    - Merge duplicates, keep most informative wording.

    JSON schema (exact):
    {
        "events": [
            {
            "id": "E1",
            "event": "string",
            "subject": "string or null",
            "object": "string or null",
            "start_time": "string or null",
            "end_time": "string or null",
            "evidence_span": "verbatim text"
            }
        ],
        "temporal_triples": [
            {
            "event1_id": "E#",
            "temp_relation": "BEFORE|AFTER|DURING|CONTAINS|IDENTITY|EQUALS|OVERLAPS",
            "event2_id": "E#",
            "evidence_span": "verbatim text"
            }
        ]
    }

    Validation:
    - Every temporal_triple must reference an event id.
    - If a string contains speach quotes, escape them with backslash (\").  
    - Use ids E1, E2... in order of appearance.  
    - evidence_span <=30 words, quoted from passage.  
    - Output must be valid JSON (no trailing commas, no comments).
    '''
    user = f'Extract events and temporal relations from the following passage (document creation time: {dct}):    {text}'
    messages = [
    {"role":"system","content":sys},
    {"role":"user","content":user}
    ]
    return messages

def inference(model, tok, text, dct, max_new_tokens=2000):
    input_prompt = tok.apply_chat_template(get_prompt(text, dct), add_generation_prompt=True, tokenize=False)
    inputs = tok(input_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.2, do_sample=False)
    gen_tokens = out[0, inputs.input_ids.shape[-1]:]
    decodings = tok.decode(gen_tokens, skip_special_tokens=True)
    prediction = decodings.strip(":\n`")
    return prediction

In [46]:
test_data = get_test_data()

In [ ]:
import json
chat_preds = []
file_num = 1
for example in test_data:
    print(f"Processing example {file_num} / {len(test_data)}")
    text = " ".join([wrd for sent in example['text'] for wrd in sent])
    for inst in example['instances']:
        if inst['type'] != 'EVENT' and inst['id'] == 0:
            dct = inst['value']
    prediction = inference(model, tok, text, dct)
    chat_preds.append({"text":text, "pred":prediction})
    file_num += 1

with open("llama3.2-3B-preds.json", 'w') as json_file:
    for sample in chat_preds:
        json_file.write(json.dumps(sample)+"\n")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processing example 1 / 256


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processing example 2 / 256


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Processing example 102 / 256


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processing example 103 / 256


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processing example 104 / 256


In [5]:
import isodate

def gentext_to_iso8601(gentext: str):
    parsers = {
        isodate.parse_date,
        isodate.parse_datetime,
        isodate.parse_time,
        isodate.parse_duration,
    }
    for parser in parsers:
        try:
            output = parser(gentext)
            if output is not None:
                return output
        except Exception:
            continue
        return None
    
def get_start_end_times(event_times: list):
    for time in event_times:
        isoobj = gentext_to_iso8601(time)
        print(isoobj)

In [6]:
events = {}
times = {}
event_quins = {}
ee_trips = []
ets = {}

for instance in test_data[0]['instances']:
    instance_id = instance["id"]
    if instance["type"] == "EVENT":
        events[instance_id] = instance
    else:
        times[instance_id] = instance

for ee in test_data[0]["ee_temprels"]:
    ee_trips.append({"event1_id":ee["e1"], "temp_relation":ee["rel"], "event2_id":ee["e2"]})

for et in test_data[0]["event_times"]:
    evid = et["event"] 
    if 'value' in times[et["time"]]:
        value = times[et["time"]]['value']
    else:
        value = None
    if evid not in ets:
        ets[evid] = [value]
    else:
        ets[evid].append(value)

for eid, event in events.items():
    quint = {
        "id": eid,
        "event": event["text"],
        "subject": None,
        "object": None,
        "times": ets.get(eid, [])
    }
    event_quins[eid] = quint


In [7]:
with open("llama3.2-3B-preds.json", "r") as f:
    llama_preds=[json.loads(line) for line in f]

In [ ]:
llama_jsons = []
for fn, pred in enumerate(llama_preds):
    print(fn+1)
    try:
        json_pred = json.loads(pred['pred'].lstrip("```json\n").rstrip("\n```"))
    except json.JSONDecodeError:
        new_pred = inference(model, tok, pred['text'], max_new_tokens=6000)
        try:
            json_pred = json.loads(new_pred.lstrip("```json\n").rstrip("\n```"))
        except json.JSONDecodeError:
            json_pred = None
    llama_jsons.append(json_pred)

In [32]:
for i, thi in enumerate(llama_jsons):
    if thi is None:
        print(i)

44
64
78
79
111
191
217
227
241


In [35]:
# Inspecting example 44
#llama_preds[44]['pred']
out = inference(model, tok, llama_preds[44]['text'], max_new_tokens=8000)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [40]:
trimmed = out.lstrip("```json\n").rstrip("\n```")
json.loads(trimmed)

JSONDecodeError: Expecting value: line 37 column 24 (char 1557)

In [41]:
trimmed

'{\n  "events": [\n    {\n      "id": "E1",\n      "event": "sought for questioning",\n      "subject": "the two men",\n      "object": "in connection with the sniper slaying",\n      "start_time": null,\n      "end_time": null,\n      "evidence_span": "The FBI has interviewed and released one of two men who were sought for questioning in connection with the sniper slaying of a doctor in the Buffalo, N.Y., area, who performed abortions."\n    },\n    {\n      "id": "E2",\n      "event": "named in a bulletin",\n      "subject": "Ronald Stauber and Michael Gingrich",\n      "object": "in a bulletin sent by the FBI office in Cleveland to police departments across the country",\n      "start_time": null,\n      "end_time": null,\n      "evidence_span": "The two men, Ronald Stauber and Michael Gingrich, were named in a bulletin sent by the FBI office in Cleveland to police departments across the country."\n    },\n    {\n      "id": "E3",\n      "event": "played down the likelihood",\n     